# Rubrik adapter'ı — eğitim

Ürünün rubrik motorunu (`/analysis/run`) çalıştıran adapter. Tek adapter iki
rubriği birden öğreniyor: `startup-investability` (9 kriter) ve
`digital-marketing` (6 kriter). İkisi farklı kriterler soruyor ama **aynı
davranışı** istiyor — şemayı doldur, vakadan alıntıla, ve vaka bir kriterde
sessizse bunu söyle.

Temel modelin ölçülmüş hali aşağıda, **1. bölümde** — ve orada duruyor çünkü
buraya yazılan bir sayı ölçüm değil, hatıradır. Bu tablonun önceki hali
gemma-2-2b-it'in `absent_rate 0 / schema_valid 0` sonucunu taşıyordu; base
Qwen3-4B'ye geçtiğinde sayılar geçmedi, yazı geçti.

**Bu notebook ölçmüyor, eğitiyor.** Tam ölçüm `rubric-eval` notebook'unda ve
orada taban ile adapter *aynı oturumda* koşuyor — aynı kütüphane sürümleri,
kesin karşılaştırılabilir sayılar. Bir Kaggle oturumu 12 saatle sınırlı ve
ikisini bir arada koşmak sekiz saate dayanıyordu; ölçümün uzaması yüzünden beş
saatlik eğitimi kaybetmek istemiyoruz.

Aşağıdaki tek istisna **ucuz taban kapısı**: 20 satırlık hızlı bir kontrol.
Amacı sayı üretmek değil, eğitime hiç başlamamak gereken durumu yakalamak —
Flutter v8'de temel model işi zaten yapıyordu ve bu, eğitim bittikten sonra
anlaşıldı.

In [ ]:
import glob, json, os, shutil, sys
import torch

assert torch.cuda.is_available(), "GPU acik degil - Settings > Accelerator > GPU T4"
cap = torch.cuda.get_device_capability(0)
print("GPU:", torch.cuda.get_device_name(0), "sm_%d%d" % cap)
print("bellek: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

# Fail here, in five seconds, rather than after an 8 GB download. A P100 is
# sm_60: Kaggle's torch build does not support it at all, and bitsandbytes needs
# sm_75 for 4-bit NF4. The Flutter run landed on one because kernel-metadata
# omitted machine_shape, and the error arrived half an hour in wearing a
# different mask.
assert cap >= (7, 5), (
    f"sm_{cap[0]}{cap[1]} yetersiz - 4-bit NF4 icin T4 (sm_75) gerekiyor. "
    "Settings > Accelerator > GPU T4 x2")

In [ ]:
# Qwen3 icin transformers >= 4.51 gerekiyor; Kaggle imaji eskiyse sessizce
# 'unknown architecture' ile duser.
!pip -q install -U "transformers>=4.51" "peft>=0.11" "bitsandbytes>=0.43" "accelerate>=0.30" datasets 2>&1 | tail -2
import transformers, peft, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__, "| bnb", bitsandbytes.__version__)
# torchao kaldiriliyor, yukseltilmiyor. peft'in LoRA dispatcher'i sardigi her
# kuantize OLMAYAN Linear icin is_torchao_available() soruyor ve o fonksiyon
# uyumsuz surumde False donmek yerine ImportError firlatiyor. Kaggle imaji
# 0.10.0 tasiyor, peft ('peft>=0.11' artik 0.20'ye cozuluyor) >0.16.0 istiyor.
#
# Tuzak fp16 kolunda: 4-bit'te bitsandbytes kendi Linear4bit'ini once
# eslestirdigi icin dispatcher'a hic varilmiyor. colab-pilot-eval bunu bir kez
# odedi ve cozdu; buraya tasinmadigi icin rubric-curve-eval ayni duvara carpti
# — taban olcumu bittikten sonra, adapter gecisinin ilk saniyesinde.
#
# Silmek find_spec'i None yapar ve kontrol False doner, ki dogru cevap odur:
# torchao nicemlemesi kullanmiyoruz. Yukseltmek torch'u da suruklerdi.
!pip -q uninstall -y torchao 2>&1 | tail -1


In [ ]:
def find_mount(slug, marker):
    """Locate one input mount by the dataset/kernel slug in its path.

    Not by filename. A kernel attached with kernel_sources contributes the whole
    of its /kaggle/working, which for the training run includes its own copies
    of the data files and the scripts — so searching for `rubric_eval.jsonl`
    finds two mounts and picks between them by luck. The slug is the only thing
    that distinguishes them, and it appears in the path.

    Recursive on top of that, because the mount depth is not a promise: the same
    dataset has appeared directly under /kaggle/input and, on the next run, one
    level deeper under /kaggle/input/datasets.
    """
    hits = [p for p in glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
            if slug.split("/")[-1] in p]
    assert hits, (f"'{slug}' bagli degil (aranan: {marker}). "
                  f"Kaggle > Notebook > Add Input, ve surumun islenmesi bitmis olmali.")
    return os.path.dirname(sorted(hits, key=len)[0])


for root, dirs, files in os.walk("/kaggle/input"):
    print(root, "->", sorted(files)[:4], "..." if len(files) > 4 else "")
    if root.count("/") > 6:
        dirs.clear()

In [ ]:
WORK = "/kaggle/working"
DATA = find_mount("emrahik/rubric-dataset", "rubric_train.jsonl")
print("veri seti:", DATA)

os.makedirs(f"{WORK}/data", exist_ok=True)
for f in os.listdir(DATA):
    dst = f"{WORK}/data/{f}" if f.endswith(".jsonl") else f"{WORK}/{f}"
    shutil.copy(f"{DATA}/{f}", dst)
os.chdir(WORK)
print(sorted(os.listdir(WORK)))
print(sorted(os.listdir(f"{WORK}/data")))

## 1. Taban kapısı — ucuz kontrol

20 satır, contrast yok. Bu hücre sayı üretmek için değil, **eğitime hiç
başlamamak gereken durumu yakalamak** için.

Bu base üzerinde ölçülmüş hali (`out/base_gate.json`, 2026-07-29):

| ölçüm | Qwen3-4B-Instruct-2507 | gemma-2-2b-it (eski hat) |
|---|---|---|
| `absent_rate` | **0.89** | 0 |
| `schema_valid` | **0.95** | 0 |
| `present_score_mae` | **0.77** | ölçülmedi |
| `hallucinated_quotes` | **0.013** | ölçülmedi |

Sağdaki sütun `peft/README.md`'nin Gemma ölçümü ve uzun süre buraya da
yazılıydı. Bu notebook'un ilk koşusu, taban zaten işi yaptığı halde eğitime
girdi: kapı JSON'u yazıyordu, kimse okumuyordu.

Yani öğretilecek davranış artık `absent_rate` değil — taban kanıt yokluğunu
zaten söylüyor. Kalan boşluk **puanın kendisi**: kanıtın orada olduğunu kabul
edip bandı kaçırıyor. Kapı bunu kontrol ediyor.

In [ ]:
import json, subprocess, sys

BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"

# subprocess.run, `!` degil. `!`'in cikis kodu hicbir yere gitmez: bu
# notebook'un ilk kosusunda egitim adim 0'da CUDA OOM ile oldu, hucre devam
# etti, ve Kaggle kernel'i COMPLETE kaydetti — geriye kanit olarak yalnizca bos
# bir out/rubric-v1 kaldi.
r = subprocess.run([sys.executable, "rubric_eval.py", "--base-only",
                    "--data", "data/rubric_eval.jsonl",
                    "--base-model", BASE_MODEL,
                    "--limit", "20",
                    "--out", "out/base_gate.json"])
assert r.returncode == 0, f"taban kapisi coktu (exit {r.returncode}) — log yukarida"

gate = json.load(open("out/base_gate.json"))["base"]
print(json.dumps(gate, indent=2))

In [ ]:
# Bu kosunun ne satin aldigi, baslamadan once beyan ediliyor. Burada olmayan
# bir bosluk, bu adapter'in kapatabilecegi bir bosluk degil.
MAE_FLOOR = 0.30          # bunun altindaysa puan zaten yeterince iyi
QUOTE_FLOOR = 0.005       # uydurma alinti pratikte sifirsa ogretilecek sey yok

mae = gate["present_score_mae"]
worth_it = (mae is not None and mae > MAE_FLOOR) or gate["hallucinated_quotes"] > QUOTE_FLOOR

print(f"present_score_mae   {mae:.2f}  (kapi: > {MAE_FLOOR})")
print(f"hallucinated_quotes {gate['hallucinated_quotes']:.1%}  (kapi: > {QUOTE_FLOOR:.1%})")
print(f"absent_rate         {gate['absent_rate']:.1%}  — taban, dusurulmemeli")
print(f"schema_valid        {gate['schema_valid']:.1%}  — taban, dusurulmemeli")

assert worth_it, (
    "Taban her iki hedefte de zaten yeterli. DUR: bes saat egitmeden once "
    "hangi davranisin ogretilecegini yeniden tanimla — Flutter v8 tam olarak "
    "burada tavana carpti ve bunu egitim bittikten sonra ogrendi.")
print("\nkapi gecildi — egitilecek olculmus bir bosluk var")

## 2. Eğitim

1600 satır, 3 epoch, effective batch 16 → ~300 optimizer adımı.

`max-seq-len 2560` ölçülerek seçildi (`measure_tokens.py`): karışımın en uzun
satırı 2477 token, p95 2308. 2048 satırların %14'ünü kırpar ve kırpma soldan
olduğu için o satırlar cevabını korur, vakasının başını kaybeder — yani modele
hiç görmediği kanıta atıf yapmayı öğretir, gayet normal görünen bir loss'la.

`PYTORCH_ALLOC_CONF=expandable_segments:True` boilerplate değil. İlk koşu adım
0'da düştü: 14.56 GB'ın 2.54 GB'ı boşken 2.58 GB istendi, ve o istek Qwen3'ün
151.936'lık vocab'ının 2477 token üzerindeki logits tensörü. Marj 40 MB, ve
PyTorch'un elinde 1.07 GB ayrılmış-ama-kullanılmayan bellek vardı —
fragmentasyon. Yine düşerse sıradaki kolluk `--max-seq-len 2308` (p95, %5
kırpılır) değil **T4 x2**: kırpma soldan olduğu için sessizce vakanın başını
atar, ve yukarıdaki paragraf onun neden pahalı olduğunu anlatıyor.

In [ ]:
import os, subprocess, sys

env = dict(os.environ, PYTORCH_ALLOC_CONF="expandable_segments:True",
           PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")

r = subprocess.run([sys.executable, "train_qlora_qwen.py",
                    "--train", "data/rubric_train.jsonl",
                    "--eval", "data/rubric_eval.jsonl",
                    "--out-dir", "out/rubric-v1",
                    "--max-seq-len", "2560",
                    "--epochs", "3", "--grad-accum", "16"], env=env)
assert r.returncode == 0, f"egitim coktu (exit {r.returncode}) — log yukarida"

# Cikis kodu 0 yetmiyor: kaydedilecek agirliklarin varligi ayrica kontrol
# edilmeli, yoksa Save Version bos bir dizini eval notebook'una girdi diye verir.
assert os.path.exists("out/rubric-v1/adapter_model.safetensors"), \
    "egitim bitti ama adapter yazilmamis — out/rubric-v1 bos"

## 3. Çıktı

Adapter `out/rubric-v1/` altında, birkaç on MB. Bu koşu **Save Version** ile
kaydedilmeli: `rubric-eval` notebook'u adapter'ı `kernel_sources` üzerinden
buradan alıyor.

Loss sonuç değil. Sonuç `rubric-eval`'in ürettiği sayılar.

In [ ]:
f = "out/rubric-v1/train_metrics.json"
if os.path.exists(f):
    print(json.dumps(json.load(open(f)), indent=2, ensure_ascii=False))

!du -sh out/rubric-v1 2>/dev/null
!ls -la out/rubric-v1